# El barrido de ruido — el cuaderno que lo corre

Este cuaderno corre la **curva de degradación**: una sola transferencia (`config.NOISE_TRANSFER`), un solo brazo de referencia (`B`, el piso), sobre los cinco niveles declarados de `config.NOISE_LEVELS`. No es una campaña más chica: es una forma distinta, por eso escribe bajo `kind="curve"` y no `"campaign"`.

Los techos se leen **una sola vez**, antes del bucle, y la misma lectura viaja a los cinco niveles — leerlos adentro del bucle daría los mismos números hoy y dejaría el barrido a merced de un registro que cambie a mitad de corrida. Y no se buscan acá: `search-pilot` (`Benchmark_Ceiling_Search.ipynb`) es lo que los busca; este cuaderno sólo los lee de vuelta del disco.


> **La escala de salida y la de entrada son dos lecturas distintas, y el paso que ejecuta este cuaderno (`barrido_de_ruido`, declarado como `noise-sweep`) ya se niega dos veces antes de abrirlo: una vez si la escala configurada no es la del ensayo, y otra vez si el registro de techos a la escala que este cuaderno lee todavía no existe.**
>
> `ES_ENSAYO` (`config.is_pilot_scale()`) decide DÓNDE escribe este barrido — el árbol de ensayo o el de la corrida completa — la misma lectura que ya usan la búsqueda y la campaña.
>
> `ENTRADA` (`config.upstream_pilot_scale()`) decide DE QUÉ ARCHIVO se leen los techos: la propia escala del recorrido en el recorrido local, y siempre la COMPLETA cuando el entorno marca un ensayo remoto — un ensayo remoto no puede consumir las salidas de otro ensayo, porque eso prueba que el cable lleva corriente y no prueba nada sobre la corrida real que va a seguirlo.


In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [2]:
# The repository was resolved by the cell above -- the forge's own, travelling
# byte for byte -- and this one only uses it. `REPOSITORY` is still the name
# every cell below reads, so nothing below changes.
import os
import sys
from pathlib import Path

REPOSITORY = ROOT
sys.path.insert(0, str(REPOSITORY / "src"))
print("repository:", REPOSITORY)

repository: /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation


In [3]:
from dataclasses import replace

from MIL_CREDA_Benchmark import config, harness

# La escala de SALIDA: dónde escribe este barrido. La misma lectura que ya
# usan la búsqueda y la campaña -- dos constantes, `EPOCHS` y `SEEDS`, leídas
# una sola vez y en un solo lugar.
ES_ENSAYO = config.is_pilot_scale()

# La escala de ENTRADA: de qué archivo se leen los techos que la búsqueda ya
# dejó. `config.upstream_pilot_scale()` y no `ES_ENSAYO`: son dos preguntas
# distintas que sólo coinciden en el recorrido LOCAL -- un ensayo remoto corre
# a escala reducida (`ES_ENSAYO=True`) y tiene que consumir lo que la búsqueda
# dejó a escala COMPLETA (`ENTRADA=False`), porque el archivo que la corrida
# real va a abrir es el completo.
ENTRADA = config.upstream_pilot_scale()

print("escala de salida:", "ensayo" if ES_ENSAYO else "completa")
print("escala de entrada:", "ensayo" if ENTRADA else "completa")


escala de salida: ensayo
escala de entrada: ensayo


/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Los techos, leídos una sola vez

Tres lecturas del mismo registro, ninguna repetida adentro del bucle: el agrupado, el pick por transferencia, y una nota de dónde salieron los dos -- el gemelo exacto de `contamination.source_note`, para que una tabla de ensayo no se pueda leer como una completa.

Nunca `harness.with_ceilings_in_force`: sin `ceilings.json` esa llamada ES la búsqueda completa entera, unas nueve horas y media que un barrido de ruido no puede lanzar por accidente. Este cuaderno sólo lee lo que la búsqueda ya dejó.


In [4]:
NOTA_TECHOS = harness.search_source_note(pilot=ENTRADA)
print(NOTA_TECHOS)

TECHOS = config.ceilings_on_record(pilot=ENTRADA)
TECHOS_POR_TRANSFERENCIA = config.ceilings_by_transfer_on_record(pilot=ENTRADA)

print("techos:", TECHOS)
print("por transferencia:", TECHOS_POR_TRANSFERENCIA)


**Estos techos son de un ENSAYO** (3 épocas), porque no hay búsqueda completa. El protocolo pide 20 épocas: no se citan como resultados, ni en el informe, ni en el resumen, ni en conversación.
techos: {'milcreda': 0.010796012633645448}
por transferencia: {'milcreda': {'M->U': 0.7117782412223816, 'U->M': 0.010796012633645448, 'M->S': 0.0005247933940780908, 'S->M': 0.0008875530880758854, 'U->S': 0.008911638421286783, 'S->U': 0.10294579502332135}}


## La corrida

Un solo brazo (`B`, el piso) sobre una sola transferencia (`config.NOISE_TRANSFER`), nivel por nivel. La biblioteca no se toca: `harness.campaign()` corre exactamente como corre para la campaña completa, sólo que acotada a un brazo, una transferencia y `kind="curve"`.


In [5]:
DISPOSITIVO = harness.resolve_device()

# La reducción base, construida UNA vez: los techos que viajan a los cinco
# niveles son el MISMO objeto, nunca releído. Cada nivel sólo cambia
# `labelNoise` -- `replace()` conserva la referencia de `ceilings`/
# `ceilingsByTransfer` en vez de copiarla.
BASE = harness.Reduction(kind="curve", pilot=ES_ENSAYO,
                         ceilings=TECHOS,
                         ceilingsByTransfer=TECHOS_POR_TRANSFERENCIA)

corridos = []
for nivel in config.NOISE_LEVELS:
    reduccion = replace(BASE, labelNoise=nivel)
    resumen = harness.campaign(reduccion, DISPOSITIVO, arms=["B"],
                               transfers=[config.NOISE_TRANSFER],
                               progress=print)
    corridos.append(f"{nivel:g}")
    print(f"nivel {nivel:g}: {len(resumen.get('grid', {}))} transferencia(s) escrita(s)")

print("niveles corridos:", corridos)


Ceilings in force: {'milcreda': 0.010796012633645448}


  M->U seed 0: 1 arms  target 0.583-0.583
nivel 0: 1 transferencia(s) escrita(s)
Ceilings in force: {'milcreda': 0.010796012633645448}


  M->U seed 0: 1 arms  target 0.528-0.528
nivel 0.1: 1 transferencia(s) escrita(s)
Ceilings in force: {'milcreda': 0.010796012633645448}


  M->U seed 0: 1 arms  target 0.667-0.667
nivel 0.2: 1 transferencia(s) escrita(s)
Ceilings in force: {'milcreda': 0.010796012633645448}


  M->U seed 0: 1 arms  target 0.667-0.667
nivel 0.3: 1 transferencia(s) escrita(s)
Ceilings in force: {'milcreda': 0.010796012633645448}


  M->U seed 0: 1 arms  target 0.333-0.333
nivel 0.4: 1 transferencia(s) escrita(s)
niveles corridos: ['0', '0.1', '0.2', '0.3', '0.4']


In [6]:
# El sello: contra qué código corrió este barrido. Sin él, un barrido viejo
# y uno recién generado se ven idénticos y el viejo se sigue creyendo.
from MIL_CREDA_Benchmark import report_digest

print(report_digest.stamp())


SOURCES-SHA256 39656385de5db5a9b97c8454c261c2b302db65416c828291a9886659685944f9
